In [ ]:
symbols = {"NVDA","META","BYND","CHGG","AAPL","BRK-B","JNJ","AMZN","GOOGL","TSLA","AMD","PLTR","RIVN","COIN","AI","UPST","FSLY","OPEN","HOOD","SPY","QQQ","XLF","XLE","ARKK","SPY"} # diverse data

import yfinance as yf
import functions
import pandas as pd
from PIL import Image
from IPython.display import display
import random
from datetime import datetime
import warnings
import math
# loop through symbols, use predictTest, use frequencies as seconds, use 1mo,3mo,6mo,1y,2y,5y as range

warnings.simplefilter(action='ignore', category=FutureWarning)
charts = functions.Charts()

def distribute(values: list, error: float):
    intensity = error*0.01
    nudged = [v + random.uniform(-intensity, intensity) for v in prev_values]
    nudged = [max(v, 0.001) for v in nudged]
    total = sum(nudged)
    return [v / total for v in nudged]

ranges = ["2022-01-01","2025-11-30"]
dates = pd.date_range(start=ranges[0], end=ranges[1])
volatileWeights = []
stableWeights = []

In [53]:
for symbol in symbols:
    print(symbol)
    beta = yf.Ticker(symbol).info.get("beta",0)
    prices = yf.download(symbol, start=ranges[0], end=ranges[1], progress=False)["Close"]
    weight = [0.2,0.2,0.2,0.2,0.2] #"duration":[weight,freq] - freq in seconds
    
    for i, date in enumerate(dates):
        realistic = False
        prevWeight = [0.2,0.2,0.2,0.2,0.2]
        prevError = 9999.0
        while not realistic:
            tests:list = distribute(prevWeight,prevError)
            bias = {90:[float(tests[0]), "ME"], 180:[float(tests[1]), "ME"], 365:[float(tests[2]), "D"], 730:[float(tests[3]), "W"], 1825:[float(tests[4]), "YS"]}
            guess = round(charts.projectTestDay(ticker=symbol, weights=str(bias), today=date),2)
            actual = round(prices.iloc[i][0],2)
            error = abs(actual-guess)
            if error < prevError:
                prevWeight = tests
                prevError = error
            print(error, prevWeight, prevError)
            if math.isclose(guess, actual, abs_tol=max(actual*0.01,0.1)):
                print(symbol, date, guess, actual)
                print(beta, tests)
                if beta > 1.2:
                    volatileWeights.append(tests)
                else:
                    stableWeights.append(tests)
                realistic = True

NVDA
41.12 [0.3575779400528714, 0.0010257180907886363, 0.1826506940412733, 0.2645095033190047, 0.19423614449606202] 41.12
26.939999999999998 [np.float64(0.23095251361733243), np.float64(0.012793869293794589), np.float64(0.33692015800368497), np.float64(0.17100908988370034), np.float64(0.24832436920148765)] 26.939999999999998
17.669999999999998 [np.float64(0.15053624091584406), np.float64(0.008952165246193577), np.float64(0.28629312342138513), np.float64(0.2833611650849624), np.float64(0.2708573053316148)] 17.669999999999998
14.749999999999998 [np.float64(0.12502399775179288), np.float64(0.008210969718171767), np.float64(0.2832059885027957), np.float64(0.23461034435995673), np.float64(0.348948699667283)] 14.749999999999998
13.129999999999999 [np.float64(0.1108242957343435), np.float64(0.008100038486774859), np.float64(0.2809305585052648), np.float64(0.207193478601157), np.float64(0.39295162867245986)] 13.129999999999999
12.049999999999997 [np.float64(0.09250649728517055), np.float64(0.0

KeyboardInterrupt: 